In [3]:
# NER model comparison: does a bigger model beat DistilBERT? (Colab, one cell)
#
# Answers "did you try a bigger model?" with the same data, the same split, the same
# training settings as the final DistilBERT run (run 4, section 10.2), and 3 seeds per
# model so the differences can be compared against seed-to-seed noise.
#
# How to run:
#   1. Colab -> Runtime -> Change runtime type -> T4 GPU. CHECK THIS FIRST: the cell stops
#      if no GPU is found (the first NER run silently trained on CPU).
#   2. Upload data/processed/ner_colab_export.json (made by ner_model/export_for_colab.py).
#   3. Paste this whole file into one cell and run it. About 30 to 45 minutes on a T4.
#   4. It downloads ner_comparison_runs.csv and ner_comparison_per_type.csv at the end.
#
# Protocol (no selection on test): each run picks its best epoch on dev. All runs are
# reported, so the test set is never used to choose anything.
# Tokens are lowercased for every model. Uncased models lowercase anyway, so their results
# are unchanged; DeBERTa-v3 is cased, and upper-case text would otherwise be split into
# many rare sub-word pieces.

!pip install -q vcs-versioning
!pip install -q transformers datasets seqeval accelerate sentencepiece

import gc, json, shutil, time
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import (AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))

MODELS = [
    # (name on the Hub, fp16). DeBERTa-v3 is run in fp32 because it can produce NaN losses in fp16.
    ("distilbert-base-uncased", True),   # 66M parameters: the current model, rerun for a seed spread
    ("bert-base-uncased", True),         # 110M
]
SEEDS = [42, 43, 44]

with open("ner_colab_export.json") as f:
    data = json.load(f)
label_list = data["label_list"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}
test_commodities = [r["commodity_type"] for r in data["test"]]
print(f"{len(data['train'])} train / {len(data['dev'])} dev / {len(data['test'])} test sentences")


def build(records):
    return Dataset.from_dict({"tokens": [[t.lower() for t in r["tokens"]] for r in records],
                              "ner_tags": [r["ner_tags"] for r in records]})


raw = DatasetDict({"train": build(data["train"]), "dev": build(data["dev"]), "test": build(data["test"])})


def to_tags(preds, labels):
    true_tags, pred_tags = [], []
    for pred, label in zip(preds, labels):
        tl, tp = [], []
        for p, l in zip(pred, label):
            if l != -100:
                tl.append(id2label[int(l)])
                tp.append(id2label[int(p)])
        true_tags.append(tl)
        pred_tags.append(tp)
    return true_tags, pred_tags


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    t, p = to_tags(np.argmax(logits, axis=2), labels)
    return {"f1": f1_score(t, p)}


runs, per_type = [], []
for model_name, fp16 in MODELS:
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_and_align(examples):
        enc = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, labels in enumerate(examples["ner_tags"]):
            prev, aligned = None, []
            for wid in enc.word_ids(batch_index=i):
                aligned.append(-100 if wid is None or wid == prev else labels[wid])
                prev = wid
            all_labels.append(aligned)
        enc["labels"] = all_labels
        return enc

    ds = raw.map(tokenize_and_align, batched=True, remove_columns=["tokens", "ner_tags"])

    for seed in SEEDS:
        set_seed(seed)
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(label_list), id2label=id2label, label2id=label2id)
        out_dir = f"./cmp_{model_name.split('/')[-1]}_{seed}"
        args = TrainingArguments(
            output_dir=out_dir, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
            learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
            num_train_epochs=15, weight_decay=0.01, load_best_model_at_end=True,
            metric_for_best_model="f1", fp16=fp16, seed=seed, report_to="none", logging_steps=10)
        trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["dev"],
                          processing_class=tokenizer, data_collator=DataCollatorForTokenClassification(tokenizer),
                          compute_metrics=compute_metrics,
                          callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
        t0 = time.time()
        trainer.train()
        minutes = (time.time() - t0) / 60

        evals = [h for h in trainer.state.log_history if "eval_f1" in h]
        best = max(evals, key=lambda h: h["eval_f1"])
        out = trainer.predict(ds["test"])
        t, p = to_tags(np.argmax(out.predictions, axis=2), out.label_ids)
        rep = classification_report(t, p, output_dict=True, zero_division=0)

        row = {"model": model_name, "seed": seed, "best_epoch": int(best["epoch"]),
               "epochs_run": int(evals[-1]["epoch"]), "dev_f1": round(best["eval_f1"], 4),
               "test_f1": round(f1_score(t, p), 4), "test_precision": round(precision_score(t, p), 4),
               "test_recall": round(recall_score(t, p), 4),
               "cause_factor_f1": round(rep.get("CAUSE_FACTOR", {}).get("f1-score", 0.0), 4),
               "minutes": round(minutes, 1)}
        for c in sorted(set(test_commodities)):
            idx = [i for i, x in enumerate(test_commodities) if x == c]
            row[f"test_f1_{c}"] = round(f1_score([t[i] for i in idx], [p[i] for i in idx]), 4)
        runs.append(row)
        for label, m in rep.items():
            if label.endswith("avg"):
                continue
            per_type.append({"model": model_name, "seed": seed, "label": label,
                             "f1": round(m["f1-score"], 4), "support": int(m["support"])})
        print(f"\n>>> {model_name} seed {seed}: dev {row['dev_f1']:.3f} (epoch {row['best_epoch']}), "
              f"test {row['test_f1']:.3f}, {minutes:.1f} min\n")

        del trainer, model
        shutil.rmtree(out_dir, ignore_errors=True)
        gc.collect()
        torch.cuda.empty_cache()

runs_df = pd.DataFrame(runs)
per_type_df = pd.DataFrame(per_type)
runs_df.to_csv("ner_comparison_runs.csv", index=False)
per_type_df.to_csv("ner_comparison_per_type.csv", index=False)

print("\nALL RUNS")
print(runs_df[["model", "seed", "best_epoch", "dev_f1", "test_f1", "test_precision", "test_recall",
               "cause_factor_f1", "minutes"]].to_string(index=False))

print("\nSUMMARY (mean ± standard deviation over seeds)")
summary = runs_df.groupby("model", sort=False).agg(
    dev_f1_mean=("dev_f1", "mean"), dev_f1_sd=("dev_f1", "std"),
    test_f1_mean=("test_f1", "mean"), test_f1_sd=("test_f1", "std"),
    test_f1_min=("test_f1", "min"), test_f1_max=("test_f1", "max"),
    cause_factor_f1_max=("cause_factor_f1", "max"), minutes_mean=("minutes", "mean"))
print(summary.round(3).to_string())

print("\nPER-TYPE TEST F1 (mean over seeds)")
pivot = per_type_df.pivot_table(index="label", columns="model", values="f1", aggfunc="mean")
pivot["support"] = per_type_df.groupby("label")["support"].first()
print(pivot[[m for m, _ in MODELS] + ["support"]].round(3).to_string())

from google.colab import files
files.download("ner_comparison_runs.csv")
files.download("ner_comparison_per_type.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.2/123.2 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
GPU: Tesla T4
760 train / 134 dev / 156 test sentences


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


>>> microsoft/deberta-v3-base seed 44: dev 0.000 (epoch 1), test 0.000, 1.0 min


ALL RUNS
                    model  seed  best_epoch  dev_f1  test_f1  test_precision  test_recall  cause_factor_f1  minutes
  distilbert-base-uncased    42           7  0.6050   0.6160          0.5923       0.6417              0.0      1.9
  distilbert-base-uncased    43           6  0.5872   0.5965          0.5685       0.6274              0.0      2.0
  distilbert-base-uncased    44           9  0.5996   0.6150          0.5744       0.6619              0.0      3.0
        bert-base-uncased    42           6  0.6128   0.6211          0.5701       0.6821              0.0      2.8
        bert-base-uncased    43          10  0.6052   0.6270          0.5785       0.6845              0.0      4.6
        bert-base-uncased    44           6  0.6044   0.6122          0.5552       0.6821              0.0      3.4
microsoft/deberta-v3-base    42           1  0.0000   0.0000          0.0000       0.0000       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>